# Sistema especialista para conferência de reservas de ingressos

**Aluno:** Gabriel Lima - 38180812    
**Disciplina:** Inteligência Artificial, Linguagens Formais e Autômatos  
**Professora:** Kadidja Valéria

## Estudo inicial

1. **O que é aprendizado de máquina?** É quando um modelo aprende padrões a partir de exemplos e usa o que aprendeu para fazer previsões em novos casos. Neste trabalho, usei uma árvore de decisão para prever a faixa de ocupação final de uma sessão.

2. **O que caracteriza o aprendizado supervisionado?** O modelo treina com exemplos que já têm uma resposta conhecida. Aqui, cada sessão simulada tem dados de vendas antecipadas e uma faixa de ocupação final calculada na simulação.

3. **O que são atributos e rótulos?** Os atributos são as informações que o modelo recebe. Usei a ocupação atual da sessão, os ingressos vendidos nos últimos sete dias e os dias restantes até o espetáculo. O rótulo é `faixa_final`, que pode ser **Baixa** (abaixo de 50%), **Média** (de 50% a menos de 80%) ou **Alta** (80% ou mais).

4. **Qual é a diferença entre classificação e regressão?** A classificação prevê uma categoria, como ocupação baixa, média ou alta. A regressão prevê um valor numérico, como a taxa final de ocupação em porcentagem. Neste trabalho, usei classificação porque o modelo prevê uma faixa.

5. **Por que separar treino e teste?** O conjunto de treino serve para a árvore aprender. O conjunto de teste serve para verificar como ela se sai com exemplos que ainda não viu. Separei 75% das sessões simuladas para treino e 25% para teste.

**Previsão de ocupação das sessões do Instituto Noah**

Este trabalho foi inspirado no sistema de ingressos que desenvolvi para o Instituto Noah. Como as compras começam semanas antes do espetáculo, a equipe pode acompanhar as vendas e decidir se precisa reforçar a divulgação de alguma sessão.

A árvore de decisão usa a ocupação atual, as vendas dos últimos sete dias e os dias restantes para prever se a ocupação final será baixa, média ou alta. O sistema usa essa previsão para recomendar uma ação à equipe.

O BI da Mostra Estações registrou 59 ingressos na sessão das 18h30 e 64 na sessão das 20h30. Esses números mostram os resultados finais de uma edição. Para treinar e testar a árvore, usei sessões simuladas, pois o BI não apresenta o histórico diário de vendas de várias sessões.

**Criar os dados simulados**

In [6]:
import random
import pandas as pd

random.seed(42)
sessoes = []

for _ in range(300):
    capacidade = 95
    dias_restantes = random.randint(1, 21)
    vendidos_agora = random.randint(8, 94)
    vendas_ultimos_7_dias = random.randint(0, min(vendidos_agora, 24))

    # Simula vendas futuras usando o ritmo recente e uma variação.
    ritmo_diario = vendas_ultimos_7_dias / 7
    variacao = random.uniform(0.5, 1.5)
    vendas_futuras = round(ritmo_diario * dias_restantes * variacao)
    vendidos_no_final = min(capacidade, vendidos_agora + vendas_futuras)

    ocupacao_final = vendidos_no_final / capacidade

    if ocupacao_final < 0.50:
        faixa_final = "Baixa"
    elif ocupacao_final < 0.80:
        faixa_final = "Média"
    else:
        faixa_final = "Alta"

    sessoes.append({
        "capacidade": capacidade,
        "vendidos_agora": vendidos_agora,
        "ocupacao_atual": vendidos_agora / capacidade,
        "vendas_ultimos_7_dias": vendas_ultimos_7_dias,
        "dias_restantes": dias_restantes,
        "vendidos_no_final": vendidos_no_final,
        "faixa_final": faixa_final
    })

dados = pd.DataFrame(sessoes)
dados.to_csv("sessoes_simuladas_noah.csv", index=False)

display(dados.head())
print(dados["faixa_final"].value_counts())

,capacidade,vendidos_agora,ocupacao_atual,vendas_ultimos_7_dias,dias_restantes,vendidos_no_final,faixa_final
0,95,22,0.231579,0,21,22,Baixa
1,95,36,0.378947,4,8,42,Baixa
2,95,19,0.200000,18,18,62,Média
3,95,19,0.200000,6,1,20,Baixa
4,95,11,0.115789,8,20,27,Baixa


faixa_final
Alta     133
Baixa     90
Média     77
Name: count, dtype: int64


**Treinar a árvore de decisão**

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

entradas = ["ocupacao_atual", "vendas_ultimos_7_dias", "dias_restantes"]
classes = ["Baixa", "Média", "Alta"]

X = dados[entradas]
y = dados["faixa_final"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

modelo = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo.fit(X_treino, y_treino)

previsoes = modelo.predict(X_teste)

print(f"Acurácia na base simulada: {accuracy_score(y_teste, previsoes):.1%}")
matriz = confusion_matrix(y_teste, previsoes, labels=classes)

display(pd.DataFrame(
    matriz,
    index=["Real: Baixa", "Real: Média", "Real: Alta"],
    columns=["Prevista: Baixa", "Prevista: Média", "Prevista: Alta"]
))

Acurácia na base simulada: 76.0%


,Prevista: Baixa,Prevista: Média,Prevista: Alta
Real: Baixa,17,4,2
Real: Média,3,14,2
Real: Alta,0,7,26


**Aplicar as quatro regras**

In [8]:
def analisar_sessao(vendidos, vendas_ultimos_7_dias,
                   dias_restantes, vagas_outra_sessao):

    capacidade = 95  # Capacidade adotada para a simulação.

    if dias_restantes is None:
        return "Quantos dias faltam para o espetáculo?"

    vagas = capacidade - vendidos

    exemplo = pd.DataFrame(
        [[vendidos / capacidade, vendas_ultimos_7_dias, dias_restantes]],
        columns=entradas
    )
    previsao = modelo.predict(exemplo)[0]

    if previsao == "Baixa" and dias_restantes >= 7:
        acao = "R1: reforce a divulgação da sessão e reavalie em uma semana."
    elif previsao == "Baixa":
        acao = "R2: faça uma divulgação de última hora para essa sessão."
    elif previsao == "Alta" and vagas <= 10 and vagas_outra_sessao > 0:
        acao = "R3: dê mais destaque à outra sessão, que ainda tem vagas."
    else:
        acao = "R4: mantenha a divulgação e acompanhe o ritmo das vendas."

    return f"Ocupação final prevista: {previsao}\n{acao}"

**Testar os casos da atividade**

In [9]:
print("Caso 1 — Poucas vendas e tempo para divulgar")
print(analisar_sessao(15, 3, 14, 40))

print("\nCaso 2 — Vendas em andamento")
print(analisar_sessao(55, 8, 7, 25))

print("\nCaso 3 — Sessão perto de lotar")
print(analisar_sessao(90, 15, 7, 30))

print("\nCaso extra — Informação ausente")
print(analisar_sessao(20, 5, None, 30))

Caso 1 — Poucas vendas e tempo para divulgar
Ocupação final prevista: Baixa
R1: reforce a divulgação da sessão e reavalie em uma semana.

Caso 2 — Vendas em andamento
Ocupação final prevista: Média
R4: mantenha a divulgação e acompanhe o ritmo das vendas.

Caso 3 — Sessão perto de lotar
Ocupação final prevista: Alta
R3: dê mais destaque à outra sessão, que ainda tem vagas.

Caso extra — Informação ausente
Quantos dias faltam para o espetáculo?


## Análise dos casos

1. **Poucas vendas e tempo para divulgar.** Com 15 ingressos vendidos e 14 dias até o espetáculo, a árvore previu ocupação baixa. A regra R1 recomendou reforçar a divulgação da sessão e olhar os resultados de novo depois de uma semana.
2. **Vendas em andamento.** Com 55 ingressos vendidos e sete dias restantes, a previsão foi de ocupação média. A regra R4 recomendou manter a divulgação e acompanhar as vendas.
3. **Sessão perto de lotar.** A árvore previu ocupação alta. Na simulação, essa sessão tinha cinco lugares livres e a outra tinha 30. A regra R3 recomendou dar mais destaque à outra sessão.
4. **Informação ausente.** Como o número de dias restantes não foi informado, o sistema perguntou quantos dias faltavam para o espetáculo.

No terceiro caso, o raciocínio para frente começa com a previsão de ocupação alta e a quantidade de vagas nas duas sessões. Esses dados acionam a R3. Para explicar o raciocínio para trás, parto da recomendação de destacar a outra sessão e verifico as três condições da R3: previsão alta, até dez vagas nessa sessão e vagas disponíveis no outro horário.